# Evaluation Analysis — Results Visualization & Comparison

In [ ]:
import sys, json
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
results_dir = Path('../results')

## Load Results

In [ ]:
with open(results_dir / 'baseline_classification_report.json') as f:
    baseline = json.load(f)

with open(results_dir / 'classification_report.json') as f:
    distilbert = json.load(f)

with open(results_dir / 'ragas_scores.json') as f:
    ragas = json.load(f)

print('Baseline weighted F1:', round(baseline['weighted avg']['f1-score'], 4))
print('DistilBERT weighted F1:', round(distilbert['weighted avg']['f1-score'], 4))

## Per-Class F1 Comparison

In [ ]:
intents = ['account_access', 'billing_issue', 'cancellation_request',
           'general_feedback', 'product_inquiry', 'technical_support']

b_f1 = [baseline.get(i, {}).get('f1-score', 0) for i in intents]
d_f1 = [distilbert.get(i, {}).get('f1-score', 0) for i in intents]

x = range(len(intents))
width = 0.35
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar([i - width/2 for i in x], b_f1, width, label='TF-IDF + LR', color='steelblue')
ax.bar([i + width/2 for i in x], d_f1, width, label='DistilBERT', color='coral')
ax.set_xticks(list(x))
ax.set_xticklabels(intents, rotation=25, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1: Baseline vs DistilBERT')
ax.legend()
ax.axhline(0.89, ls='--', color='green', alpha=0.6, label='Target (0.89)')
plt.tight_layout()
plt.savefig(results_dir / 'f1_comparison.png', dpi=150)
plt.show()

## RAGAS Scores

In [ ]:
agg = ragas.get('aggregate', {})
for metric, stats in agg.items():
    print(f'{metric}:')
    for k, v in stats.items():
        print(f'  {k}: {v:.4f}')
    print()

## RAGAS Score Distribution

In [ ]:
per_query = pd.DataFrame(ragas.get('per_query', []))
metrics = [c for c in per_query.columns if c in ('faithfulness', 'answer_relevancy')]

if metrics:
    fig, axes = plt.subplots(1, len(metrics), figsize=(7 * len(metrics), 4))
    if len(metrics) == 1:
        axes = [axes]
    for ax, metric in zip(axes, metrics):
        per_query[metric].dropna().hist(bins=20, ax=ax, color='teal', edgecolor='white')
        ax.axvline(per_query[metric].mean(), color='red', ls='--', label=f'Mean={per_query[metric].mean():.3f}')
        ax.set_title(metric.replace('_', ' ').title())
        ax.set_xlabel('Score')
        ax.legend()
    plt.tight_layout()
    plt.savefig(results_dir / 'ragas_distribution.png', dpi=150)
    plt.show()

## Training Curves

In [ ]:
from PIL import Image
curve_path = Path('../models/distilbert/training_curves.png')
if curve_path.exists():
    img = Image.open(curve_path)
    plt.figure(figsize=(14, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print('Training curves not yet generated.')

## Comparison Table

In [ ]:
table_path = results_dir / 'comparison_table.md'
if table_path.exists():
    print(table_path.read_text())
else:
    print('Comparison table not yet generated. Run run_evaluation.py first.')